# Readability

## Goals
### 1. Validate the efficacy of the CLEAR dataset for assessing readabililty
### 2. Compare the efficacy of two novel frameworks for assessing readability against a ground truth CAREC Readability Score:
- Framework 1 is an original approach to calculating readability as a combination of syntax, lexical difficulty (AKA word/vocab difficulty), grammar, and lexical diversity. We will utilize an established readability score called Flesch Reading Ease to assess the syntactic component of our text data. Lexical difficulty will be assessed via counting word frequency in a separate corpus of text. Grammar will be assessed via the grammar-checking API, and lexical diversity will also be calculated via API. Together, these four components will be combined and scaled to a number between 0 and 100 to create a novel readability scoring framework.
- Framework 2 is an LLM-as-a-judge approach - prompt Claude 3.5 Haiku to score text based on its own definition of readability on a scale of 0 - 100.

## A Statement About Readability
Readability is a major focus of computational linguistics and linguistic research. There is simply no one "accepted" method of judging readability - only popular ways and less popular ways. [William DuBay's "The Principles of Readability" (2004)](https://files.eric.ed.gov/fulltext/ED490073.pdf) better discusses this lack of consensus. Understandably, it is a complex topic that is seemingly abstract and has many interested stakeholders contributing to discourse, meaning there is continual research to this day that attempts to optimize scoring systems of readability, and define what it is that readability consists of. Just to paint a picture of why there is much discussion about the topic, at the turn of the century, there was a large research push due to poorly written child-safety seat instructions that resulted in injury and death. This explains why there is not a well-known LLM readability evaluation framework, and that we can proceed with creating a novel way to score it for the needs of the project. Readability has a large focus in educational circles, where it defines how to classify reading grade levels. There are ethical considerations that must be made when defining readability, where we must factor in cultural and dialectic contexts. Therefore, we will avoid complexity and seek to understand readability in the scope of the project's needs - what makes a chat agent's responses easy to understand? A common definition used is the intersection of syntax, word complexity, grammar, and word diversity. A sentence should be constructed with natural syntactic reading flow, the vocabulary should not be unnecessarily challenging, the sentence should adhere to basic grammar rules, and it should attempt to vary word choices. The latest readability scores can analyze hundreds of text features, but we determine that most of those boil down to the aforementioned four categories.

## Readability Scores
- **Flesch Reading Ease (1975)**: Perhaps the most commonly used readability score to this day, it considers a ratio of syllables per word to syllables per sentence. It is understood as a score between 0 and 100, where higher scores correlate with an elementary level reading skill, and a lower score correlate with a post-grad reading skill (though it is formulaically possible to score lower or higher than this range, we will scale all values to be between 0 and 100). We can see the limitation that a piece of text with a difficult vocabulary but simple syntactic structure will score artificially very readable. However, this makes Flesch Reading Ease a suitable stand-in to consider syntax in our original readability scoring framework.
- **New Dale-Chall (1995)**: New Dale-Chall addresses the issue of lexical difficulty by way of a 3000-word dictionary of "easy words." It then considers any word not on this list to be a "difficult" word, and adds a penalty for the number of difficult words to the score. This is a more nuanced approach to readability than Flesch Reading Ease, but it still is not robust enough to recognize varying difficulty of individual words, or detect grammar mistakes.
- **CAREC: Crowdsourced Algorithm of Reading Comprehension (2019)**: On the bleeding edge of scores is CAREC, which utilizes modern NLP techniques to identify readability across hundreds of text features. CAREC is trained on a very large dataset and is the closest we get to emulating the behaviorial responses of a reader that would identify a text as truly "readable" or not, in the most comprehensive estimation of "readability." Consider this example: *"Cat and Dog walk. They walk in their village. Then they see an egg. The egg is in the grass. The egg is alone in the grass. The egg is all alone. Cat and Dog walk to a bird. They ask the bird, "Is this your egg?" But the bird says, "No, that is not my egg. Ask the owl. Maybe it is his egg ..."* This piece of text that requires an elementary level reading skill may score well with Flesch and Dale-Chall, but CAREC may assess it to have low lexical diversity which would would mark it as relatively difficult to read. You, as the reader, could also recognize the monotonous word repitition as unpleasant to read. This presents the distinction between a piece of text being *unchallenging* to read vs it having the genuine trait of readability. Thus, we are again reminded that the definition of "readability" can be borderline subjective depending on the score used. However, CAREC is the most modern score, so it is the most sensible to select as a ground truth score. It will be the best encapsulation of the idea of readability to compare our novel readability evaluation framework and Claude readability evaluation framework to. Readers would not wish a LLM to use exclusively first-grade vocabulary as a method of improving its quality of readability, so CAREC's consideration of lexical diversity and numerous other features are a welcome addition. A higher CAREC score indicates less readability.

## Assessing readability using the CLEAR dataset
- The **CLEAR** dataset was developed by the CommonLit organization and Georgia State University to support readability research. It contains 5000 literature excerpts of varying difficulty, which we will consider as a one-to-one substitution for the responses of a chat agent.
- Contains objective readability scores such as Flesch Reading Ease, New Dale-Chall, CAREC, and many other noteable readability scoring systems. Some of these scores, like Flesch-Kincaid Grade Level, are companion scores used in educational contexts to determine the grade level of a text. This dataset, on release, was the center of a Kaggle competition that sought to produce the most robust readabilty classifier, and the predicted readability scores from the top submissions were also annotated in the dataset, though we will trust more in the original readability score we calculate ourselves for maximum explainability.
- read further: 
  - https://seantrott.substack.com/p/measuring-the-readability-of-texts
  - https://www.commonlit.org/blog/introducing-the-clear-corpus-an-open-dataset-to-advance-research-28ff8cfea84a/
  - https://docs.google.com/spreadsheets/d/1sfsZhhP2umXXtmEP_NRErxLuwgN98TyH7LWOq3j07O0/edit?gid=971821388#gid=971821388

## Steps
1. Clean the dataset, saving the CAREC (Ground Truth Score) and Flesch Reading Ease Score (Syntax Score) rows
2. Factor in lexical difficulty as a measure of readability
3. Factor in grammar as a measure of readability
4. Factor in lexical diverity as a measure of readability
5. Combine the above four factors into a novel readability score
6. Prompt Claude to score the readability of each excerpt based on its own definition of readability
7. Compare the novel readability framework and Claude readability framework to the Ground Truth (CAREC) score

In [1]:
%%capture
!pip install pandas
!pip install nltk
!pip install language_tool_python
!pip install lexicalrichness
!pip install anthropic
!python -m nltk.downloader punkt

In [31]:
import pandas as pd

#Lexical Difficulty
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.probability import FreqDist

#Grammar Checker
from language_tool_python import LanguageTool
import spacy
from scipy.stats import boxcox

#Lexical Diversity
from lexicalrichness import LexicalRichness

#Claude
import boto3
import json
import os
import time
from dotenv import load_dotenv

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('words')
nltk.download('reuters')
nltk.download('brown')
nltk.download('gutenberg')
nltk.download('names')
nltk.download('webtext')
nltk.download('nps_chat')
nltk.download('wordnet')
nltk.download('movie_reviews')

[nltk_data] Downloading package punkt to /Users/colby/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /Users/colby/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /Users/colby/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package words to /Users/colby/nltk_data...
[nltk_data]   Package words is already up-to-date!
[nltk_data] Downloading package reuters to /Users/colby/nltk_data...
[nltk_data]   Package reuters is already up-to-date!
[nltk_data] Downloading package brown to /Users/colby/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package gutenberg to /Users/colby/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!
[nltk_data] Downloading package names to /Users/colby/nltk_data...
[nltk_data]   Package names is already up-to-date!
[nltk_data] Downloading pack

True

# Step 1: Clean the CLEAR dataset

In [195]:
clear = pd.read_csv('../data/CLEAR_readability_original.csv')
clear.head().T

,0,1,2,3,4
ID,400,401,402,403,404
Last Changed,NaN,NaN,NaN,NaN,NaN
Author,Carolyn Wells,Carolyn Wells,Carolyn Wells,CHARLES KINGSLEY,Charles Kingsley
Title,Patty's Suitors,Two Little Women on a Holiday,Patty Blossom,THE WATER-BABIES\nA Fairy Tale for a Land-Baby,HOW THE ARGONAUTS WERE DRIVEN INTO THE UNKNOWN...
Anthology,NaN,NaN,NaN,NaN,The Heroes\n or Greek Fairy Tales for my...
URL,http://www.gutenberg.org/cache/epub/5631/pg563...,http://www.gutenberg.org/cache/epub/5893/pg589...,http://www.gutenberg.org/cache/epub/20945/pg20...,http://www.gutenberg.org/files/25564/25564-h/2...,http://www.gutenberg.org/files/677/677-h/677-h...
Source,gutenberg,gutenberg,gutenberg,gutenberg,gutenberg
Pub Year,1914.0,1917.0,1917.0,1863.0,1889.0
Category,Lit,Lit,Lit,Lit,Lit
Location,mid,mid,mid,mid,mid


Save only the Flesch Reading Ease to process into a syntax score, and the CAREC score as the most robust ground truth readability score to compare our novel and LLM-as-a-judge frameworks against.

In [196]:
clear = clear[['Excerpt', 'CAREC', 'Flesch-Reading-Ease']].rename(columns = {'CAREC': 'Ground Truth Score', 'Flesch-Reading-Ease': 'Syntax Score'})
clear.head()

,Excerpt,Ground Truth Score,Syntax Score
0,When the young people returned to the ballroom...,0.12102,81.70
1,"All through dinner time, Mrs. Fayre was somewh...",0.04921,80.26
2,"As Roger had predicted, the snow departed as q...",0.10172,79.04
3,Mr. Grimes was to come up next morning to Sir ...,0.07491,44.77
4,And outside before the palace a great garden w...,0.06356,68.07


Apply two transformations to the CAREC scores that will turn it into our Ground Truth Score. Firstly, it makes sense that a higher number indicates more readability and a lowest number indicates less readability, so we will have to flip the interpretation of the CAREC scores so that lower Ground Truth Scores means a more difficult text. Secondly, we will scale the CAREC scores to a 0-100 scale for easier comparison.

Use min-max scaling on the Flesch Reading Ease scores to normalize them to the Syntax Score on a scale of 0 - 100, and the inverse 0 - 100 scale for the CAREC scores to convert them into the Ground Truth Score. Now all scores can be interpreted in the same way, where scores closer to 0 are more difficult to read and scores closer to 100 are easier to read.

In [16]:
def min_max_normalize(col):
    """Normalize column values to 0 - 100 Scale"""
    clear[col] = (clear[col] - clear[col].min()) / (clear[col].max() - clear[col].min()) * 100

def inverse_normalize(col):
    """Normalize and invert column values to 0 - 100 Scale"""
    clear[col] = 100 - (clear[col] - clear[col].min()) / (clear[col].max() - clear[col].min()) * 100


In [198]:
inverse_normalize('Ground Truth Score')
min_max_normalize('Syntax Score')

### EDA

View the apparent "hardest" excerpts which will have the lowest Ground Truth Scores, and the "easiest" excerpts which will have the highest Ground Truth Scores.

In [199]:
ground_truth_sorted = clear.sort_values(by="Ground Truth Score", ascending=True)[['Excerpt', 'Ground Truth Score', 'Syntax Score']]
ground_truth_sorted.head()

,Excerpt,Ground Truth Score,Syntax Score
602,The Internet Protocol (IP) is the principal co...,0.000000,42.756258
831,"Solubility is the property of a solid, liquid,...",10.503567,44.888827
3005,"The new fact which I have observed is, that in...",12.267614,40.756538
1924,"The principal subjects, concerning which Presi...",12.982347,20.696406
732,Organic chemistry is a chemistry subdiscipline...,13.777795,29.632219


In [200]:
ground_truth_sorted = clear.sort_values(by="Ground Truth Score", ascending=False)[['Excerpt', 'Ground Truth Score', 'Syntax Score']]
ground_truth_sorted.head()

,Excerpt,Ground Truth Score,Syntax Score
3301,Some weeks passed by before Nellie was allowed...,100.000000,76.821424
3368,"Dear Mr. Santa Claus—Please, sir, could you no...",94.321200,87.092714
2588,"The little pig said, ""Ready! I have been and c...",94.288653,75.877500
3342,"One day mousie was so hungry, that he made bol...",93.822580,80.925745
3245,"One day, when I was harnessing him, he spied a...",93.541374,79.492379


Already in this analysis we can see some disagreement between the Flesch Reading Ease and CAREC scores, lending towards the more robust nature of the CAREC score.

Now to view the distribution of the Syntax Scores:

In [201]:
syntax_sorted = clear.sort_values(by="Syntax Score", ascending=True)
syntax_sorted.head()

,Excerpt,Ground Truth Score,Syntax Score
2562,"It is further agreed by the parties hereto, th...",16.764308,0.000000
2057,"What began, during the springtime of my actual...",30.190595,2.202489
504,Environmental science is an interdisciplinary ...,32.918034,5.355894
695,Molecular nanotechnology (MNT) is a technology...,19.296464,9.928681
314,"Lowell was a man of wide learning, and has a p...",41.622403,14.263739


In [202]:
syntax_sorted = clear.sort_values(by="Syntax Score", ascending=False)
syntax_sorted.head()

,Excerpt,Ground Truth Score,Syntax Score
1327,Cat and Dog walk. They walk in their village. ...,29.857314,100.000000
1328,Cat and Dog open the door. They open the door ...,51.787481,98.944204
3262,"""But why?"" yelped the pup, as the maid threw a...",81.498985,97.958328
3263,"The horse and the cow, in great grief, came an...",81.927303,97.056356
4419,A man tied his horse to a tree and went into a...,64.751601,96.301217


# Step 2: Estimating Lexical Difficulty

Adapted a non-ML lexical difficulty solution to read a full body of text and return the average word difficulty. We will not consider stop words ("and", "I", e.t.c) or common names as legitimate words, and the returned difficulty scale will be from 0 - 2500, where that number is the frequency of times a word is spotted from the corpus of text, capped at 2500. The Lexical Difficulty Score will be generated as the average word difficulty of the entire text, considering all legitimate words.

Text Corpus (from Omer Duskin):
- movie_reviews
  - This corpus contains movie reviews and is useful for assessing the difficulty of words encountered in the context of film-related content.
- reuters
  - comprising news articles from the Reuters newswire, this corpus helps in evaluating words commonly used in news reports.
- brown
  - A general corpus of American English text, ideal for assessing the difficulty of words in everyday language.
- gutenberg
  - This corpus consists of public domain books, including classic literature and other high-level content, making it valuable for words encountered in literature.
- webtext
  - A collection of informal language collected from web forums, overheard conversations in New York, advertising, wine reviews, and other assorted sources.
- nps_chat
  - A collection of instant messaging chat sessions, offering insights into words used in informal digital communication.

References:
- https://python.plainenglish.io/estimating-word-difficulty-in-english-using-python-a-practical-guide-8f6812de5122
- https://github.com/dusking/medium_src/blob/main/src/a01_word_difficulty.py

In [212]:
MAX_SCORE = 2500

class WordDifficulty:
    """Estimate the difficulty level of a given English word.

    This class provides methods to estimate the difficulty level of an English word
    based on its usage frequency in various corpora and other linguistic transformations.

    Attributes:
        stopwords (set): A set of common English stopwords.
        words (set): A set of common English words.
        names (set): A set of common names.
        word_freq (dict): A dictionary containing frequency distributions of words in different corpora.
        lemmatizer (WordNetLemmatizer): An instance of WordNetLemmatizer for lemmatization.
    """

    def __init__(self):
        """Initialize the WordDifficulty class and download required NLTK resources."""
        self.stopwords = set(nltk.corpus.stopwords.words('english'))
        self.words = set(nltk.corpus.words.words())
        self.names = set(nltk.corpus.names.words())

        all_synsets = list(nltk.corpus.wordnet.all_synsets())
        all_wordnet_words = []
        for synset in all_synsets:
            all_wordnet_words.extend(synset.lemma_names())
        self.wordnet_words = set(all_wordnet_words)

        self.word_freq = {
            "words": FreqDist(nltk.corpus.words.words()),
            "wordnet_words": FreqDist(all_wordnet_words),
            
            "movie_reviews": FreqDist(nltk.corpus.movie_reviews.words()),
            "reuters": FreqDist(nltk.corpus.reuters.words()),
            "brown": FreqDist(nltk.corpus.brown.words()),
            "gutenberg": FreqDist(nltk.corpus.gutenberg.words()),
            "webtext": FreqDist(nltk.corpus.webtext.words()),
            "nps_chat": FreqDist(nltk.corpus.nps_chat.words()),
        }
        self.lemmatizer = WordNetLemmatizer()

    """Estimate the average lexical difficulty of a piece of text.

    Args:
        text (str): The text to evaluate.

    Returns:
        float or None: The difficulty score of the text (average lexical difficulty score).
    """
    def score_text_difficulty(self, text):
        sum = 0
        words = text.split()
        num_words = 0

        for word in words:
            word = word.lower()
            eval_result = self.eval_word(word)
            if self.is_a_word(word, eval_result) and (word not in self.stopwords or word not in self.names):
                num_words += 1
                sum += self.score_word_difficulty(word, eval_result)
        
        if num_words == 0:
            return MAX_SCORE
        return sum / num_words

    def score_word_difficulty(self, word, eval_result):
        """Estimate the difficulty of a word based on its usage frequency.

       Args:
           word (str): The word to evaluate.
           eval_result (dict): A pre-evaluated word frequency dictionary.

       Returns:
           int or None: The difficulty score of the word, or None if the word is not found in any corpus.
       """
        sum_eval = sum(eval_result.values())
        if sum_eval > MAX_SCORE:
            return MAX_SCORE # easy words like "hi" that are not stopwords can artificially score in the thousands
        return sum_eval

    def eval_word(self, word):
        """Evaluate the word's frequency in various corpora.

        Args:
            word (str): The word to evaluate.

        Returns:
            dict: A dictionary containing the word's frequency in different corpora.
        """
        word = word.lower()
        return {corpus: freq[word] for corpus, freq in self.word_freq.items()}

    def is_a_word(self, word, eval_result=None):
        """Check if a word is a valid English word.

        Args:
            word (str): The word to check.
            eval_result (dict): A pre-evaluated word frequency dictionary (optional).

        Returns:
            bool: True if the word is a valid English word; otherwise, False.
        """
        if any(value in self.stopwords for value in [word, word.title()]):
            return True
        if any(value in self.words for value in [word, word.title()]):
            return True
        if not eval_result:
            eval_result = self.eval_word(word)
        sum_eval = sum(eval_result.values())
        if sum_eval == 0:
            return False
        non_movie_reviews_eval = sum(value for key, value in eval_result.items() if key not in ["movie_reviews"])
        if non_movie_reviews_eval == 0 and word.title() in self.names:
            return False
        return True
    
word_difficulty = WordDifficulty()

In [213]:
clear["Lexical Difficulty Score"] = clear["Excerpt"].apply(word_difficulty.score_text_difficulty)
clear.head()

,Excerpt,Ground Truth Score,Syntax Score,Lexical Difficulty Score
0,When the young people returned to the ballroom...,62.327501,77.394770,1789.555556
1,"All through dinner time, Mrs. Fayre was somewh...",71.676301,76.387918,1843.911290
2,"As Roger had predicted, the snow departed as q...",64.840129,75.534890,1825.781250
3,Mr. Grimes was to come up next morning to Sir ...,68.330469,51.573207,1799.938462
4,And outside before the palace a great garden w...,69.808103,67.864634,1663.704545


Let's look at the smallest and largest scores, and view the distribution of scores (to see if we need to correc the MAX_SCORE value):

In [214]:
text_difficulty = clear.sort_values(by="Lexical Difficulty Score", ascending = True)
text_difficulty.head()

,Excerpt,Ground Truth Score,Syntax Score,Lexical Difficulty Score
695,Molecular nanotechnology (MNT) is a technology...,19.296464,9.928681,1160.609375
1028,Who loves splashing? The Irrawaddy Dolphin lov...,51.605218,72.395469,1171.155738
504,Environmental science is an interdisciplinary ...,32.918034,5.355894,1214.894309
659,"Magnetic resonance imaging (MRI), nuclear magn...",51.433370,40.623689,1215.881119
782,"Radiosurgery is surgery using radiation, that ...",13.861115,24.045588,1220.638462


In [215]:
text_difficulty = clear.sort_values(by="Lexical Difficulty Score", ascending = False)
text_difficulty.head()

,Excerpt,Ground Truth Score,Syntax Score,Lexical Difficulty Score
374,There was a king long ago in Ireland. He had t...,59.475082,85.547476,2240.917910
4301,Once upon a time there were three bears who li...,75.206999,82.547895,2229.534247
1404,"One day, the people of Mongu held a meeting. T...",84.807061,83.058314,2212.654930
43,This is the story of how the swallow's tail ca...,65.101807,84.253950,2185.291667
4392,Now it came to pass in the days when the judge...,63.878040,70.591526,2184.766871


In [216]:
clear['Lexical Difficulty Score'].describe()

count    4724.000000
mean     1781.275552
std       168.804758
min      1160.609375
25%      1674.281384
50%      1791.730340
75%      1902.331169
max      2240.917910
Name: Lexical Difficulty Score, dtype: float64

Rescale Lexical Difficulty Score to 0 - 100

In [217]:
min_max_normalize('Lexical Difficulty Score')
clear['Lexical Difficulty Score'].describe()

count    4724.000000
mean       57.452677
std        15.625606
min         0.000000
25%        47.548639
50%        58.420437
75%        68.658329
max       100.000000
Name: Lexical Difficulty Score, dtype: float64

# Step 3: Estimating Grammar Quality

Utilize [LanguageTool's](https://languagetool.org/)'s `language_tool_python` library to count grammatical errors for each excerpt. Count the number of grammatical errors, count the number of words in each excerpt, and calculate the Grammar Score as the ratio of grammatical errors to word length, then min-max scale from 0 - 100. Additionally rely on `spacy` package to reduce wrongly identified spelling by identifying proper nouns.

Initialize the server connection and request American English rules, and load the nlp model for the English language.

In [265]:
tool = LanguageTool('en-US')
nlp = spacy.load("en_core_web_sm")

In [299]:
def get_proper_nouns(text):
    """Extract proper nouns from a given text."""
    doc = nlp(text)
    proper_nouns = []
    for token in doc:
        if token.pos_ == 'PROPN':
            proper_nouns.append(token.text)
    return proper_nouns

def score_grammar(text):
    """Find ratio of grammar errors to number of words in a given text."""
    matches = tool.check(text)

    num_words = len(text.split())
    num_errors = 0
    for error in matches:
        if 'Possible spelling mistake found' in error.message:
            #British spelling and proper nouns are not spelling mistakes
            flagged_word = error.context[error.offset : error.offset + error.errorLength]
            if('is British English' in error.message or flagged_word in get_proper_nouns(text)):
                continue      
        num_errors += 1
    return num_errors / num_words

In [300]:
clear["Grammar Score"] = clear["Excerpt"].apply(score_grammar)
clear.head()

,Excerpt,Ground Truth Score,Syntax Score,Lexical Difficulty Score,Grammar Score
0,When the young people returned to the ballroom...,62.327501,77.394770,58.219125,0.011173
1,"All through dinner time, Mrs. Fayre was somewh...",71.676301,76.387918,63.250626,0.005917
2,"As Roger had predicted, the snow departed as q...",64.840129,75.534890,61.572398,0.024096
3,Mr. Grimes was to come up next morning to Sir ...,68.330469,51.573207,59.180231,0.006289
4,And outside before the palace a great garden w...,69.808103,67.864634,46.569582,0.018293


Close the server connection

In [304]:
tool.close()

In [327]:
clear['Grammar Score'].describe()

count    4724.000000
mean        0.012873
std         0.017025
min         0.000000
25%         0.000000
50%         0.006667
75%         0.017167
max         0.240260
Name: Grammar Score, dtype: float64

The grammar scores are very highly skewed towards zero, so we will take a Box-Cox Transformation to normalize, then inverse scale from 0 - 100, where higher scores indicate better grammar quality.

In [329]:
clear['Grammar Score'], _ = boxcox(clear['Grammar Score'] + 1)
clear['Grammar Score'].describe()

count    4724.000000
mean        0.007064
std         0.005913
min         0.000000
25%         0.000000
50%         0.005652
75%         0.011456
max         0.019975
Name: Grammar Score, dtype: float64

In [330]:
inverse_normalize('Grammar Score')
clear['Grammar Score'].describe()

count    4724.000000
mean       64.636885
std        29.603738
min         0.000000
25%        42.648786
50%        71.702284
75%       100.000000
max       100.000000
Name: Grammar Score, dtype: float64

Now view the worst and best grammar scores:

In [342]:
grammar_sorted = clear.sort_values(by="Grammar Score", ascending=True)
grammar_sorted.head()

,Excerpt,Ground Truth Score,Syntax Score,Lexical Difficulty Score,Grammar Score
4590,"Yay! It is bath time for Chunnu and Munnu,\nTo...",24.410248,79.995805,44.961596,0.000000
1544,So Naipei sang a song asking what he had been ...,58.152372,76.415886,60.476362,0.015710
2959,There are five species of salmon (Oncorhynchus...,46.283133,60.942526,40.326751,0.025342
1700,"Soon, Rini's father brought her a puppy. \nIt ...",78.046399,79.408474,70.596974,0.051539
4500,Little Cicada loved to sing. But her song was ...,59.868250,85.848133,45.518808,0.073606


In [343]:
grammar_sorted = clear.sort_values(by="Grammar Score", ascending=False)
grammar_sorted.head()

,Excerpt,Ground Truth Score,Syntax Score,Lexical Difficulty Score,Grammar Score
20,Father had been away in the country for three ...,67.057231,69.395889,66.379213,100.0
4723,Animals are made of many cells. They eat thing...,50.899599,64.368620,65.312368,100.0
4719,The name Monarch means “king”. An adult Monarc...,60.408530,81.359250,38.175880,100.0
4718,The second state of matter we will discuss is ...,58.177108,80.422319,68.225094,100.0
4715,When you think of dinosaurs and where they liv...,56.562777,71.304713,42.634478,100.0


# Step 4: Estimate Lexical Diversity
There are numerous measures of lexical diversity. The most simple would be Type-Token Ratio (TTR), which is simply the number of unique words divided by the total number of words. More robust is the Measure of Textual Lexical Diversity (MTLD), which is the mean length of words strings that have reached a certain threshold of diversity. Use the `lexicalrichness` library to compute MTLD for each excerpt, and min-max scale to 0 - 100.

In [12]:
def score_lexical_diversity(text):
    """Find ratio of unique words to total number of words in a given text."""
    lex = LexicalRichness(text)
    return lex.mtld()

In [13]:
clear["Lexical Diversity Score"] = clear["Excerpt"].apply(score_lexical_diversity)
clear.head()

,Excerpt,Ground Truth Score,Syntax Score,Lexical Difficulty Score,Grammar Score,Lexical Diversity Score
0,When the young people returned to the ballroom...,62.327501,77.394770,58.219125,0.011173,55.795609
1,"All through dinner time, Mrs. Fayre was somewh...",71.676301,76.387918,63.250626,0.005917,100.348996
2,"As Roger had predicted, the snow departed as q...",64.840129,75.534890,61.572398,0.024096,103.460159
3,Mr. Grimes was to come up next morning to Sir ...,68.330469,51.573207,59.180231,0.006289,90.775131
4,And outside before the palace a great garden w...,69.808103,67.864634,46.569582,0.018293,64.062500


In [14]:
clear['Lexical Diversity Score'].describe()

count    4724.000000
mean       76.237193
std        25.388320
min        11.116071
25%        58.105762
50%        73.816662
75%        91.096794
max       216.000000
Name: Lexical Diversity Score, dtype: float64

Rescale the Lexical Diversity Score to 0 - 100, where higher scores indicate more lexical diversity.

In [17]:
min_max_normalize('Lexical Diversity Score')
clear['Lexical Diversity Score'].describe()

count    4724.000000
mean       31.784397
std        12.391563
min         0.000000
25%        22.934786
50%        30.602981
75%        39.037089
max       100.000000
Name: Lexical Diversity Score, dtype: float64

Now view the worst and best Lexical Diversity Scores:

In [18]:
lexical_diversity_sorted = clear.sort_values(by="Lexical Diversity Score", ascending=True)
lexical_diversity_sorted.head()

,Excerpt,Ground Truth Score,Syntax Score,Lexical Difficulty Score,Grammar Score,Lexical Diversity Score
1064,"You eat colors, when you eat fruits. Eat apple...",59.264177,83.407915,48.034625,0.012048,0.000000
1325,This is Cat. This is Dog. Cat and Dog live in ...,62.236369,93.874983,40.558393,0.037037,0.122958
1326,Cat and Dog look through the window. They look...,23.310160,81.617956,55.793346,0.005747,1.484402
1329,Dog is in his house. Dog is sitting in his hou...,52.930532,93.098867,37.522615,0.052288,2.297077
1234,A nerve is a group of special nerve cells grou...,42.134042,74.262341,42.204140,0.010989,2.378260


In [19]:
lexical_diversity_sorted = clear.sort_values(by="Lexical Diversity Score", ascending=False)
lexical_diversity_sorted.head()

,Excerpt,Ground Truth Score,Syntax Score,Lexical Difficulty Score,Grammar Score,Lexical Diversity Score
2396,"Bull, John, a fine, fat, American-beef fed ind...",46.984846,63.019158,36.969329,0.055556,100.000000
3948,After some work Tom succeeded in reducing the ...,57.086132,73.416305,58.376274,0.005263,92.331191
2475,In what does this noble disregard for appearan...,59.512837,60.460076,64.874934,0.010050,91.702619
3655,The two lads had come to a halt on the road ab...,66.490913,70.346805,56.038678,0.017045,90.797174
1484,"The cross-Atlantic flight of Charles ""Lucky Li...",49.522210,51.831912,30.550791,0.000000,89.000044


# Step 5: Calculate the Novel Readability Score

We will consider the Syntax Score, Lexical Difficulty Score, Grammary Score, and Lexical Diversity Score to be equally weighted in the Novel Readability Score. Thus, we will sum these scores and divide by 4 to get the Novel Readability Score, then min-max scale to 0 - 100.

In [20]:
clear["Novel Readability Score"] = clear[['Syntax Score', 'Lexical Difficulty Score', 'Grammar Score', 'Lexical Diversity Score']].mean(axis=1)
clear.head()

,Excerpt,Ground Truth Score,Syntax Score,Lexical Difficulty Score,Grammar Score,Lexical Diversity Score,Novel Readability Score
0,When the young people returned to the ballroom...,62.327501,77.394770,58.219125,0.011173,21.807244,39.358078
1,"All through dinner time, Mrs. Fayre was somewh...",71.676301,76.387918,63.250626,0.005917,43.552915,45.799344
2,"As Roger had predicted, the snow departed as q...",64.840129,75.534890,61.572398,0.024096,45.071416,45.550700
3,Mr. Grimes was to come up next morning to Sir ...,68.330469,51.573207,59.180231,0.006289,38.880092,37.409955
4,And outside before the palace a great garden w...,69.808103,67.864634,46.569582,0.018293,25.842158,35.073667


In [21]:
clear['Novel Readability Score'].describe()

count    4724.000000
mean       38.782427
std         6.861559
min         6.519466
25%        34.347337
50%        39.615058
75%        43.768391
max        57.917062
Name: Novel Readability Score, dtype: float64

Now view the worst and best Novel Readability Scores:

In [23]:
novel_readability_worst = clear.sort_values(by="Novel Readability Score", ascending=True)
novel_readability_worst.head()

,Unnamed: 0,Excerpt,Ground Truth Score,Syntax Score,Lexical Difficulty Score,Grammar Score,Lexical Diversity Score,Novel Readability Score,Claude Readability Score
695,695,Molecular nanotechnology (MNT) is a technology...,19.296464,9.928681,0.000000,10.801147,27.204830,11.983664,NaN
504,504,Environmental science is an interdisciplinary ...,32.918034,5.355894,5.024947,52.631859,15.684119,19.674205,NaN
1028,1028,Who loves splashing? The Irrawaddy Dolphin lov...,51.605218,72.395469,0.976236,7.042626,4.195576,21.152477,NaN
2562,2562,"It is further agreed by the parties hereto, th...",16.764308,0.000000,46.895645,23.263386,15.937124,21.524039,NaN
602,602,The Internet Protocol (IP) is the principal co...,0.000000,42.756258,10.997533,20.196829,13.803146,21.938442,NaN


In [24]:
novel_readability_best = clear.sort_values(by="Novel Readability Score", ascending=False)
novel_readability_best.head()

,Unnamed: 0,Excerpt,Ground Truth Score,Syntax Score,Lexical Difficulty Score,Grammar Score,Lexical Diversity Score,Novel Readability Score,Claude Readability Score
2042,2042,When Grandmother Lane was a little girl her fa...,91.084726,75.150329,89.208712,100.0,64.858151,82.304298,NaN
2992,2992,I will tell you what stopped him. While the du...,87.292350,88.253391,83.055226,100.0,49.300815,80.152358,NaN
2975,2975,"THIS house has no roof, no chimney, no windows...",80.145029,86.281639,75.368626,100.0,52.562231,78.553124,NaN
3359,3359,We have a new dog. His name is Bright. He is o...,93.144300,88.589009,93.341267,100.0,31.268364,78.299660,NaN
3262,3262,"""But why?"" yelped the pup, as the maid threw a...",81.498985,97.958328,92.451887,100.0,22.468708,78.219730,NaN


# Step 6: Use Claude 3.5 Haiku as an LLM-as-a-Judge Evaluation Framework for Readability

Utilize a carefully constructed prompt to Claude to evaluate the readability of each excerpt. Claude will return a score between 0 and 100, where higher scores indicate more readability. Additionally, few shot Claude with the worst and best reading passages according to the Novel Readability Score to hopefully contextualize its scoring.

Initialize connector information and parameters

In [102]:
load_dotenv()
AWS_KEY = os.getenv('AWS_KEY')
AWS_SECRET_KEY = os.getenv('AWS_SECRET_KEY')

modelId = 'arn:aws:bedrock:us-east-2:522814696964:inference-profile/us.anthropic.claude-3-5-haiku-20241022-v1:0'
region = 'us-east-2'
temperature = 0.7
max_tokens_to_generate = 7 #number probably shouldn't be longer than this


In [103]:
bedrock_runtime = boto3.client(service_name='bedrock-runtime', 
                               region_name=region, 
                               aws_access_key_id=AWS_KEY, 
                               aws_secret_access_key=AWS_SECRET_KEY)

Initialize our prompts and the Claude model

In [107]:
NUM_SCORED = len(clear)

bad_score_example = novel_readability_worst.iloc[0]['Excerpt']
bad_score = f"{novel_readability_worst.iloc[0]['Novel Readability Score']:.2f}"

good_score_example = novel_readability_best.iloc[0]['Excerpt']
good_score = f"{novel_readability_best.iloc[0]['Novel Readability Score']:.2f}"

prompt_template = f"""
On a readability scale from 0.0 to 100.0:
- {bad_score} represents the least readable text like this: "{bad_score_example}"
- {good_score} represents the most readable like this: "{good_score_example}"

Based on this scale, output only a single number between 0.0 and 100.0 that represents the readability score for this text:

{{excerpt}}

Just provide the number without any explanation.
"""

In [108]:
print(prompt_template.format(excerpt=novel_readability_worst.iloc[0]['Excerpt']))


On a readability scale from 0.0 to 100.0:
- 11.98 represents the least readable text like this: "Molecular nanotechnology (MNT) is a technology based on the ability to build structures to complex, atomic specifications by means of mechanosynthesis. This is distinct from nanoscale materials. Based on Richard Feynman's vision of miniature factories using nanomachines to build complex products (including additional nanomachines), this advanced form of nanotechnology (or molecular manufacturing) would make use of positionally-controlled mechanosynthesis guided by molecular machine systems. MNT would involve combining physical principles demonstrated by biophysics, chemistry, other nanotechnologies, and the molecular machinery of life with the systems engineering principles found in modern macroscale factories. While conventional chemistry uses inexact processes obtaining inexact results, and biology exploits inexact processes to obtain definitive results, molecular nanotechnology would em

Append Claude's responses to the Claude Readability Score Column

In [109]:
for i in range(NUM_SCORED):
    excerpt = clear.loc[i, 'Excerpt']
    print(f'Excerpt {i}: {excerpt[:50]}...')    

    try:
        readability_prompt = prompt_template.format(excerpt=excerpt)
        messages = [
            {"role": "user", "content": readability_prompt}
        ]

        body = json.dumps({
            "messages": messages,
            "temperature": temperature,
            "max_tokens": max_tokens_to_generate,
            "anthropic_version": "bedrock-2023-05-31"
        })
        
        response = bedrock_runtime.invoke_model(
            body=body, 
            modelId=modelId,
            accept="application/json", 
            contentType="application/json"
        )
        
        response_body = json.loads(response.get('body').read())
        claude_readability = response_body.get('content')[0]['text']
        
        clear.loc[i, 'Claude Readability Score'] = claude_readability        
        time.sleep(0.1)
        
    except Exception as e:
        print(f"Error processing excerpt {i}: {e}")

clear.head()

Excerpt 0: When the young people returned to the ballroom, it...
Excerpt 1: All through dinner time, Mrs. Fayre was somewhat s...
Excerpt 2: As Roger had predicted, the snow departed as quick...
Excerpt 3: Mr. Grimes was to come up next morning to Sir John...
Excerpt 4: And outside before the palace a great garden was w...
Excerpt 5: Once upon a time there were Three Bears who lived ...
Excerpt 6: Hal and Chester found ample time to take an invent...
Excerpt 7: Hal Paine and Chester Crawford were typical Americ...
Excerpt 8: On the twenty-second of February, 1916, an automob...
Excerpt 9: The boys left the capitol and made their way down ...
Excerpt 10: One day he had gone beyond any point which he had ...
Excerpt 11: It was believed by the principal men of Virginia t...
Excerpt 12: This Pedrarias was seventy-two years old. He was o...
Excerpt 13: The Emperor walked nervously up and down the long,...
Excerpt 14: The clock in a nearby church struck the hour of tw...
Excerpt 15: Aunt Abi

,Excerpt,Ground Truth Score,Syntax Score,Lexical Difficulty Score,Grammar Score,Lexical Diversity Score,Novel Readability Score,Claude Readability Score
0,When the young people returned to the ballroom...,62.327501,77.394770,58.219125,57.334588,21.807244,53.688932,64.70
1,"All through dinner time, Mrs. Fayre was somewh...",71.676301,76.387918,63.250626,74.426414,43.552915,64.404468,53.72
2,"As Roger had predicted, the snow departed as q...",64.840129,75.534890,61.572398,30.359599,45.071416,53.134576,72.4
3,Mr. Grimes was to come up next morning to Sir ...,68.330469,51.573207,59.180231,73.060867,38.880092,55.673599,50.20
4,And outside before the palace a great garden w...,69.808103,67.864634,46.569582,40.352185,25.842158,45.157140,57.40


In [123]:
clear['Claude Readability Score'] = clear['Claude Readability Score'].astype(float)

In [124]:
clear['Claude Readability Score'].describe()

count    4724.000000
mean       57.618603
std        16.017632
min        11.100000
25%        47.600000
50%        56.215000
75%        68.420000
max        95.200000
Name: Claude Readability Score, dtype: float64

In [125]:
claude_readability_worst = clear.sort_values(by="Claude Readability Score", ascending=True)
claude_readability_worst.head()

,Excerpt,Ground Truth Score,Syntax Score,Lexical Difficulty Score,Grammar Score,Lexical Diversity Score,Novel Readability Score,Claude Readability Score
3005,"The new fact which I have observed is, that in...",12.267614,40.756538,51.440425,43.713317,16.446174,38.089113,11.10
2508,Even before the ice came creeping southwestwar...,48.997552,58.802965,66.687323,75.181767,32.111770,58.195956,11.30
2862,The small volume of which we have spoken is de...,26.810915,42.085023,46.329455,26.871702,35.000965,37.571786,11.32
1974,"On April 16 M. Gukovsky, the Commissary for Fi...",33.390616,46.364145,38.057704,45.643792,50.259683,45.081331,11.72
2749,The hydrochloric acid gas passes into a vessel...,59.304536,64.732205,38.451655,77.203509,22.882069,50.817359,11.74


In [126]:
claude_readability_best = clear.sort_values(by="Claude Readability Score", ascending=False)
claude_readability_best.head()

,Excerpt,Ground Truth Score,Syntax Score,Lexical Difficulty Score,Grammar Score,Lexical Diversity Score,Novel Readability Score,Claude Readability Score
1464,"Ape meets Crocodile by the lake. ""Where do you...",52.849815,84.540624,47.501313,100.000000,8.607679,60.162404,95.2
1330,First Dog draws an oval. The oval is the body ...,60.094777,90.267096,59.840099,16.343596,4.720297,42.792772,95.2
1624,"Goat said, ""Ehe, Pig! Come with me to my house...",78.404416,89.889526,71.581317,76.783029,38.706863,69.240184,94.7
1147,Aku looked at her fufu. She looked at her soup...,67.840962,85.002098,64.225810,0.218657,8.905458,39.588006,93.2
1238,Fati was also very busy. She found small piece...,69.679217,85.918053,72.057684,5.631658,7.962272,42.892417,92.7


Upload the cleaned and updated CLEAR dataset

In [127]:
clear.to_csv('../data/CLEAR_readability_test.csv')

# Step 7: Compare the Novel Readability Score and LLM-as-a-judge to the Ground Truth Score

Crown the more accurate readability framework by comparing their Mean Absolute Error (MAE) to the Ground Truth Score, as well as the number of readability predictions each framework got correct within a 10-point margin of error.

Start by computing the difference between each score (including the composite scores of the novel readability framework) and the Ground Truth Score

In [128]:
error = pd.DataFrame()

error["Syntax Error"] = clear["Syntax Score"] - clear["Ground Truth Score"]
error["Lexical Difficulty Error"] = clear["Lexical Difficulty Score"] - clear["Ground Truth Score"]
error["Grammar Error"] = clear["Grammar Score"] - clear["Ground Truth Score"]
error["Lexical Diversity Error"] = clear["Lexical Diversity Score"] - clear["Ground Truth Score"]
error["Novel Readability Error"] = clear["Novel Readability Score"] - clear["Ground Truth Score"]
error["Claude Readability Error"] = clear["Claude Readability Score"] - clear["Ground Truth Score"]
error.head()

,Syntax Error,Lexical Difficulty Error,Grammar Error,Lexical Diversity Error,Novel Readability Error,Claude Readability Error
0,15.067269,-4.108376,-4.992913,-40.520257,-8.638569,2.372499
1,4.711617,-8.425674,2.750113,-28.123385,-7.271832,-17.956301
2,10.694761,-3.267731,-34.480530,-19.768713,-11.705553,7.559871
3,-16.757263,-9.150238,4.730398,-29.450377,-12.656870,-18.130469
4,-1.943469,-23.238521,-29.455918,-43.965945,-24.650963,-12.408103


Order the two framework/composite scores by their MAE

In [129]:
mae = error.abs().mean().sort_values()
mae

Lexical Difficulty Error     9.505975
Novel Readability Error      9.891701
Claude Readability Error    10.581687
Syntax Error                11.221199
Lexical Diversity Error     26.408496
Grammar Error               28.265712
dtype: float64

Find the number of predictions each framework/composite score got correct within a 10-point margin of error

In [130]:
correct_counts = []  

for col in error.columns:  
    count = (abs(error[col]) < 10).sum()  
    correct_counts.append((col, count))  

correct_counts.sort(key=lambda tuple: tuple[1], reverse=True)  

for col, count in correct_counts:  
    print(f"{col}: {count}/{len(clear)} ({count/len(clear)*100:.2f}%)")

Lexical Difficulty Error: 2870/4724 (60.75%)
Novel Readability Error: 2768/4724 (58.59%)
Claude Readability Error: 2605/4724 (55.14%)
Syntax Error: 2352/4724 (49.79%)
Grammar Error: 931/4724 (19.71%)
Lexical Diversity Error: 792/4724 (16.77%)
